<a href="https://colab.research.google.com/github/Onureeva/airbnb-vienna-nlp-pricing/blob/main/Reviews_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.

In [ ]:
# Import required libraries for data analysis and visualization

import pandas as pd              # for data manipulation
import numpy as np               # for numerical operations
import matplotlib.pyplot as plt  # for plotting
import seaborn as sns            # for enhanced visualizations

# Set a default aesthetic style for plots
sns.set(style="whitegrid")



In [ ]:
# import Airbnb Listing datasets for Vienna

reviews_url = 'https://data.insideairbnb.com/austria/vienna/vienna/2025-09-14/data/reviews.csv.gz'

# Load the datasets into DataFrames
reviews_df = pd.read_csv(reviews_url, compression='gzip')



In [ ]:
# Ex1 a: Display the first 10 rows

reviews_df.head(10)

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,40625,73717,2010-08-04,176849,William,Ingela is a superb host. She personally welco...
1,40625,110809,2010-10-03,222519,Kerri,Ingela was a perfect host! She gave great dire...
2,40625,206046,2011-03-22,273895,Heather,Our stay in Vienna with Ingela could not have ...
3,40625,554329,2011-09-21,254998,Fernando,Our stay in the beautiful city of Vienna was g...
4,40625,584745,2011-10-01,314952,Michael,We really enjoyed our visit and loved the very...
5,40625,756907,2011-12-01,148764,Julie,"My experience was great, with a really lovely ..."
6,40625,782765,2011-12-13,1386050,Christian,I totally agree with the reviews of the previo...
7,40625,851835,2012-01-09,1230408,Kristina & Oleg,We spent several days in Vienna in January 201...
8,40625,2510231,2012-10-05,2616415,Sue,"This was a very pleasant flat, just as describ..."
9,40625,2648798,2012-10-18,3444804,Pui Yee,Ingela was a great host. Though we did not mee...


In [ ]:
reviews_df.shape

(612430, 6)

In [ ]:
# Ex1 b: Explore columns, data types, and non-null counts

!pip install langdetect
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException
reviews_df.info()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=325d512c40bedfd8b7e4978fecfa741ceac2b727576aa5643d3928ecef812230
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612430 entries, 0 to 612429
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   listing_id     612430 non-null  int64 
 1   id             612430 non-null  int64 
 2   date           612430 non-null  object
 3   reviewer_id    612430 non-null  int64 
 4   reviewer_name  612427 non-null  object
 5   comments       612395 non-null  object
dtypes: int64(3), object(3)
memory usage: 28.0+ MB


In [ ]:
def detect_language(text):
    try:
        # Ensure the input is treated as a string, handling non-string types (like NaN)
        if pd.isna(text): # Check for NaN values
            return None
        return detect(str(text)) # Convert to string before detection
    except LangDetectException:
        return None

reviews_df['language']=reviews_df['comments'].apply(detect_language)
reviews_df['language'].value_counts()


,count
language,
en,326172
de,144875
fr,33410
es,21639
it,15574
ko,8260
nl,6777
ru,6450
tr,4557


In [ ]:
reviews_df['language'].nunique()

45

2. DATASET CLEANING

In [ ]:
reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612430 entries, 0 to 612429
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   listing_id     612430 non-null  int64 
 1   id             612430 non-null  int64 
 2   date           612430 non-null  object
 3   reviewer_id    612430 non-null  int64 
 4   reviewer_name  612427 non-null  object
 5   comments       612395 non-null  object
 6   language       609059 non-null  object
dtypes: int64(3), object(4)
memory usage: 32.7+ MB


In [ ]:
#Drop columns id, reviewer_id, reviewer_name
df=reviews_df.drop(['id', 'reviewer_id', 'reviewer_name'], axis=1)


In [ ]:
#Choose only english comments
english_df=df[df['language']=='en']

In [ ]:
english_df.shape

(326172, 4)

In [ ]:
# Remove missing and empty review texts
df = english_df.dropna(subset=['comments']).copy()
df = df[df['comments'].str.strip() != ''].copy()


In [ ]:
#delete duplicates
df = df.drop_duplicates(
    subset=['listing_id', 'comments', 'date']
).copy()

In [ ]:
#Change data format
df['date'] = pd.to_datetime(df['date'])

In [ ]:
import re

def clean_soft(text):
    """
    Minimal cleaning for transformer-based sentiment analysis.
    Preserves capitalization and punctuation.
    """
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)   # remove HTML tags
    text = re.sub(r'http\S+', ' ', text)   # remove URLs
    text = re.sub(r'\s+', ' ', text)       # normalize whitespace
    return text.strip()


def clean_hard(text):
    """
    Normalized text for TF-IDF and topic modeling.
    """
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)    # remove punctuation
    text = re.sub(r'\d+', '', text)        # remove digits
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# Preserve two versions of every review
df['comments_soft'] = df['comments'].apply(clean_soft)
df['comments_clean'] = df['comments_soft'].apply(clean_hard)

In [ ]:
# Remove extremely short reviews with fewer than 3 words
df['review_length'] = df['comments_clean'].apply(
    lambda x: len(x.split())
)

df = df[df['review_length'] > 2].copy()

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 319614 entries, 0 to 612429
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   listing_id      319614 non-null  int64         
 1   date            319614 non-null  datetime64[ns]
 2   comments        319614 non-null  object        
 3   language        319614 non-null  object        
 4   comments_soft   319614 non-null  object        
 5   comments_clean  319614 non-null  object        
 6   review_length   319614 non-null  int64         
dtypes: datetime64[ns](1), int64(2), object(4)
memory usage: 19.5+ MB


In [ ]:
df.isnull().sum()

,0
listing_id,0
date,0
comments,0
language,0
comments_soft,0
comments_clean,0
review_length,0


In [ ]:
# Save preprocessed reviews with separate text versions
df.to_csv('cleaned_reviews.csv', index=False)

In [ ]:
from google.colab import files
files.download('cleaned_reviews.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>